In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Overview
- MNIST dataset
  + Goal: estimating summation of handwritten digit images
  + Input: a set of 28x28 images
  + Output: summation of given handwritten digit images
- Defining an CNN+LSTM for MNIST dataset
- Training and evaluating an CNN+LSTM

# Import libraries

In [ ]:
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_set = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST('./data', train=False, download=True, transform=transform)
x_train = train_set.data.float().unsqueeze(1).div(255).to(device)
y_train = train_set.targets.long().to(device)
x_test = test_set.data.float().unsqueeze(1).div(255).to(device)
y_test = test_set.targets.long().to(device)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

# (1) Data

In [ ]:
class CNNLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.lstm = nn.LSTM(64 * 7, 128, batch_first=True)
        self.classifier = nn.Linear(128, 10)
    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1, 3).flatten(2)
        _, (hidden, _) = self.lstm(x)
        return self.classifier(hidden[-1])
model = CNNLSTM().to(device)
print(model)

## Original data
- Input: 48x48 image
- Output: label (0, 1,..., 9)

## Data preprocessing
- Read images from MNIST
- Compute their summation

## New data
- Input: eight 48x48 images
- Output: summation of eight input images

In [ ]:
lr = 0.01
batch_size = 128
epochs = 10

# (2) Declare model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.0)
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size)

def evaluate(loader):
    model.eval(); total_loss = 0.0; correct = 0; count = 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb); total_loss += criterion(logits, yb).item() * yb.size(0)
            correct += (logits.argmax(1) == yb).sum().item(); count += yb.size(0)
    return total_loss / count, correct / count

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
starting_time = time.time()
for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad(); loss_value = criterion(model(xb), yb); loss_value.backward(); optimizer.step()
    train_loss, train_acc = evaluate(train_loader)
    val_loss, val_acc = evaluate(test_loader)
    history['loss'].append(train_loss); history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss); history['val_accuracy'].append(val_acc)
    print(f'Epoch {epoch + 1}/{epochs}: loss={train_loss:.4f}, accuracy={train_acc:.4f}, val_accuracy={val_acc:.4f}')
print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

In [ ]:
test_loss, test_accuracy = evaluate(test_loader)
print('Test loss:', test_loss)
print('Test accuracy:', test_accuracy)

random_test_index = np.random.randint(len(test_set))
test_img = x_test[random_test_index:random_test_index + 1]
test_label = y_test[random_test_index].item()
model.eval()
with torch.no_grad():
    pred_label = model(test_img).argmax(1).item()
plt.imshow(test_img[0, 0].cpu(), cmap='gray')
plt.title(f'true label = {test_label}, predicted label = {pred_label}')
plt.show()

# (3) Train defined model
- Note that the training history including loss and accuracy will be save in 'history' variable
- In case your system runs out of memory (OOM), try to decrease batch size

### Training time
- Training time with GTX 1080 is about 26.6267 minutes for 30 epochs
- Training time with Core i7 is around 808 minutes (~13.5 hours) for 30 epochs

# (4) Evaluate trained model